# Dataset Preparation


## Imports


In [1]:
import os
import subprocess
import xml.etree.ElementTree as ET

import numpy as np

## Constants


In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Set SUMO_HOME environment variable
os.environ["SUMO_HOME"] = os.path.join(os.environ["CONDA_PREFIX"], "Lib", "site-packages", "sumo")
SUMO_HOME = os.environ["SUMO_HOME"]

# Add SUMO_HOME/bin and SUMO_HOME/tools to PATH
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "bin")
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "tools")

# Directories for files
TRAIN_DIR = "train"
TEST_DIR = "test"

# Create directories if they do not exist
for dir in [TRAIN_DIR, TEST_DIR]:
    if not os.path.exists(dir):
        os.makedirs(dir)

# Paths to network and tools
NETWORK_PATH = os.path.join("athens-osmWebWizard", "osm.net.xml.gz")
RANDOM_TRIPS_PATH = os.path.join(SUMO_HOME, "tools", "randomTrips.py")
DUAROUTER_PATH = os.path.join(SUMO_HOME, "bin", "duarouter.exe")

# Vehicle types and their probabilities
VEHICLE_TYPES_PROBABILITIES = {"car": 0.70, "ldv": 0.10, "truck": 0.05, "bus": 0.15}

# Define traffic generation volumes for 10 hours of traffic
TRAFFIC_VOLUMES = [1, 2, 6, 5, 3, 1, 2, 6, 4, 3]

## Traffic Volumes


In [3]:
# Normalize and scale raw volumes
processed_traffic_volumes = [12 / x for x in TRAFFIC_VOLUMES]

# Generate train volumes
train_traffic_volumes = list(np.array(processed_traffic_volumes))

# Generate test traffic volumes based on train volumes multiplied by a small random factor
test_traffic_volumes = [v * np.random.uniform(0.95, 1.05) for v in train_traffic_volumes]

## Random Trips


In [4]:
def generate_random_trips(traffic_volumes, output_directory):
    for hour, volume in enumerate(traffic_volumes):
        start_time = hour * 3600
        end_time = start_time + 3600
        output_file = os.path.join(output_directory, f"{hour:02d}h-trips.rou.xml")
        command = [
            "python",
            RANDOM_TRIPS_PATH,
            "-n",
            NETWORK_PATH,
            "-b",
            str(start_time),
            "-e",
            str(end_time),
            "-p",
            str(volume),
            "--edge-param",
            "main",
            "--validate",
            "-o",
            output_file,
        ]

        print("Executing:", " ".join(command))
        result = subprocess.run(command, capture_output=True, text=True)
        print(result.stdout)
        print(result.stderr)

        if os.path.exists(output_file):
            print("Created:", output_file)
        else:
            print("Failed:", output_file)


print("Generating train random trips...")
generate_random_trips(train_traffic_volumes, TRAIN_DIR)
print("Generating test random trips...")
generate_random_trips(test_traffic_volumes, TEST_DIR)

Generating train random trips...
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n athens-osmWebWizard\osm.net.xml.gz -b 0 -e 3600 -p 12.0 --edge-param main --validate -o train\00h-trips.rou.xml
Success.
Success.


Created: train\00h-trips.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n athens-osmWebWizard\osm.net.xml.gz -b 3600 -e 7200 -p 6.0 --edge-param main --validate -o train\01h-trips.rou.xml
Success.
Success.


Created: train\01h-trips.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n athens-osmWebWizard\osm.net.xml.gz -b 7200 -e 10800 -p 2.0 --edge-param main --validate -o train\02h-trips.rou.xml
Success.
Success.


Created: train\02h-trips.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n athens-osmWebWizard\osm.net.xml.gz -b 10800 -e 1440

## Duarouter Routes


In [5]:
def compute_vehicle_routes(directory):
    for filename in sorted(os.listdir(directory)):
        if not filename.endswith("trips.rou.xml"):
            continue

        trips_file = os.path.join(directory, filename)
        output_file = os.path.join(directory, filename.replace("trips", "routes"))
        command = [
            DUAROUTER_PATH,
            "-n",
            NETWORK_PATH,
            "-r",
            trips_file,
            "-o",
            output_file,
        ]

        print("Executing:", " ".join(command))
        result = subprocess.run(command, capture_output=True, text=True)
        print(result.stdout)
        print(result.stderr)

        if os.path.exists(output_file):
            print("Created:", output_file)
        else:
            print("Failed:", trips_file)


print("Computing train vehicle routes...")
compute_vehicle_routes(TRAIN_DIR)
print("Computing test vehicle routes...")
compute_vehicle_routes(TEST_DIR)

Computing train vehicle routes...
Executing: C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\bin\duarouter.exe -n athens-osmWebWizard\osm.net.xml.gz -r train\00h-trips.rou.xml -o train\00h-routes.rou.xml
Reading up to time step: 0.00
Reading up to time step: 200.00
Reading up to time step: 400.00
Reading up to time step: 600.00
Reading up to time step: 800.00
Reading up to time step: 1000.00
Reading up to time step: 1200.00
Reading up to time step: 1400.00
Reading up to time step: 1600.00
Reading up to time step: 1800.00
Reading up to time step: 2000.00
Reading up to time step: 2200.00
Reading up to time step: 2400.00
Reading up to time step: 2600.00
Reading up to time step: 2800.00
Reading up to time step: 3000.00
Reading up to time step: 3200.00
Reading up to time step: 3400.00
Reading up to time step: 3600.00
Success.


Created: train\00h-routes.rou.xml
Executing: C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\bin\duarouter.exe -n athens-osmWebWizard\osm

## Vehicle IDs


In [6]:
def update_vehicle_ids(directory):
    vehicle_id = 0
    for filename in sorted(os.listdir(directory)):
        if not filename.endswith("routes.rou.xml"):
            continue

        routes_file = os.path.join(directory, filename)
        tree = ET.parse(routes_file)
        root = tree.getroot()

        for vehicle in root.findall("vehicle"):
            vehicle.set("id", str(vehicle_id))
            vehicle_id += 1
        tree.write(routes_file)

        print("Updated:", routes_file)


print("Updating train vehicle IDs...")
update_vehicle_ids(TRAIN_DIR)
print("Updating test vehicle IDs...")
update_vehicle_ids(TEST_DIR)

Updating train vehicle IDs...
Updated: train\00h-routes.rou.xml
Updated: train\01h-routes.rou.xml
Updated: train\02h-routes.rou.xml
Updated: train\03h-routes.rou.xml
Updated: train\04h-routes.rou.xml
Updated: train\05h-routes.rou.xml
Updated: train\06h-routes.rou.xml
Updated: train\07h-routes.rou.xml
Updated: train\08h-routes.rou.xml
Updated: train\09h-routes.rou.xml
Updating test vehicle IDs...
Updated: test\00h-routes.rou.xml
Updated: test\01h-routes.rou.xml
Updated: test\02h-routes.rou.xml
Updated: test\03h-routes.rou.xml
Updated: test\04h-routes.rou.xml
Updated: test\05h-routes.rou.xml
Updated: test\06h-routes.rou.xml
Updated: test\07h-routes.rou.xml
Updated: test\08h-routes.rou.xml
Updated: test\09h-routes.rou.xml


## Vehicle Types


In [7]:
def update_vehicle_types(directory):
    for filename in sorted(os.listdir(directory)):
        if not filename.endswith("routes.rou.xml"):
            continue

        routes_file = os.path.join(directory, filename)
        tree = ET.parse(routes_file)
        root = tree.getroot()

        for vehicle in root.findall("vehicle"):
            random_vehicle_type = np.random.choice(
                list(VEHICLE_TYPES_PROBABILITIES.keys()),
                p=list(VEHICLE_TYPES_PROBABILITIES.values()),
            )
            vehicle.set("type", random_vehicle_type)
        tree.write(routes_file)

        print("Updated:", routes_file)


print("Updating train vehicle types...")
update_vehicle_types(TRAIN_DIR)
print("Updating test vehicle types...")
update_vehicle_types(TEST_DIR)

Updating train vehicle types...
Updated: train\00h-routes.rou.xml
Updated: train\01h-routes.rou.xml
Updated: train\02h-routes.rou.xml
Updated: train\03h-routes.rou.xml
Updated: train\04h-routes.rou.xml
Updated: train\05h-routes.rou.xml
Updated: train\06h-routes.rou.xml
Updated: train\07h-routes.rou.xml
Updated: train\08h-routes.rou.xml
Updated: train\09h-routes.rou.xml
Updating test vehicle types...
Updated: test\00h-routes.rou.xml
Updated: test\01h-routes.rou.xml
Updated: test\02h-routes.rou.xml
Updated: test\03h-routes.rou.xml
Updated: test\04h-routes.rou.xml
Updated: test\05h-routes.rou.xml
Updated: test\06h-routes.rou.xml
Updated: test\07h-routes.rou.xml
Updated: test\08h-routes.rou.xml
Updated: test\09h-routes.rou.xml
